# CTDenoiser — Training & Benchmark Notebook

| Part | What happens |
|------|--------------|
| **0 — Setup** | Mount Drive, clone/update repo, install deps |
| **1 — Smoke tests** | Verify all five models run end-to-end on synthetic data |
| **2 — Train** | Train all models on real TCIA LDCT data |
| **3 — Compare** | Side-by-side metric table across all models |
| **4 — Visualise** | Denoised slice grid: noisy → each model's output |
| **5 — W&B sweep** | Bayesian HP search across all architectures |

**Models included:** RED-CNN · DnCNN · U-Net · CTformer · Conditional Flow Matching (CFM)  
**Metrics:** PSNR · SSIM · RMSE · GMSD · NPS-ratio

**Prerequisites:** Run `00_build_cache.ipynb` once to download the TCIA LDCT data and build `ldct_cache.h5` on your Drive.

---
## Part 0 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'main'
REPO_ROOT = '/content/drive/MyDrive/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    os.makedirs(REPO_ROOT, exist_ok=True)
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    print('Cloned.')
else:
    # Remove stale lock files left by interrupted git/training processes
    for lock in ['.git/index.lock', '.git/refs/heads/lock']:
        lp = os.path.join(REPO_ROOT, lock)
        if os.path.exists(lp):
            os.remove(lp)
            print(f'Removed stale lock: {lock}')

%cd {REPO_ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
%cd {REPO_ROOT}
!pip install -q -r requirements.txt

In [ ]:
%cd {REPO_ROOT}
!pytest -q  # expect 11 passed, 1 skipped (HDF5 test needs h5py)

---
## Part 1 — Smoke Tests (Synthetic Data)

Verifies every model's training loop, eval, and checkpoint saving work  
end-to-end without needing real CT data. Each run takes < 30 s on CPU.

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model redcnn    --epochs 1 --synthetic-len 32
print('RED-CNN smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model dncnn     --epochs 1 --synthetic-len 32
print('DnCNN smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model unet      --epochs 1 --synthetic-len 32
print('U-Net smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model ctformer  --epochs 1 --synthetic-len 32
print('CTformer smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
# --flow-steps 5 keeps the Euler ODE fast for a smoke run (default is 20)
!python -m ctdenoiser.train --model flowmatching --epochs 1 --synthetic-len 32 --flow-steps 5
print('CFM smoke test passed.')

---
## Part 2 — Train All Models on Real Data

Copies the HDF5 cache to fast local disk, then trains all five models  
sequentially. Adjust `--epochs` and `--batch-size` to your GPU budget.

**Typical settings on a T4 GPU:**  
- `--epochs 50 --batch-size 16` → ~10 min per CNN model, ~25 min for CFM  
- `--epochs 100` for publication-quality results

In [ ]:
import os, shutil

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_CACHE):
    print('Copying cache to local disk for faster I/O...')
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Done.')

assert os.path.exists(LOCAL_CACHE), (
    'ldct_cache.h5 not found — run 00_build_cache.ipynb first, '
    'or place the file at ' + DRIVE_CACHE
)

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model redcnn \
    --h5-cache {LOCAL_CACHE} \
    --epochs 1 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model dncnn \
    --h5-cache {LOCAL_CACHE} \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model unet \
    --h5-cache {LOCAL_CACHE} \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model ctformer \
    --h5-cache {LOCAL_CACHE} \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
# --flow-steps 20   : Euler steps used at *inference* time (saved into checkpoint).
# --flow-steps-eval 5 : steps used during post-training validation.
!python -m ctdenoiser.train \
    --model flowmatching \
    --h5-cache {LOCAL_CACHE} \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4 \
    --flow-steps 20 \
    --flow-steps-eval 5

---
## Part 3 — Results Comparison Table

Loads every saved checkpoint, runs full-slice overlapped inference on the  
validation split, and prints a side-by-side metric table.

| Metric | Better |
|--------|--------|
| PSNR (dB) | ↑ higher |
| SSIM | ↑ higher |
| RMSE | ↓ lower |
| GMSD | ↓ lower (perceptual proxy) |
| NPS-ratio | ↓ lower (spectral residual) |

In [ ]:
import os, sys, torch
sys.path.insert(0, REPO_ROOT)

from pathlib import Path
from torch.utils.data import DataLoader

from ctdenoiser.data.dataset import HDF5CTDataset
from ctdenoiser.inference import overlapped_inference
from ctdenoiser.metrics import gmsd, nps_ratio, psnr, rmse, ssim
from ctdenoiser.models import CTformer, DnCNN, FlowMatching, REDCNN, UNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH  = 64

# For a thorough final comparison use more steps; keep at 5 for a quick check.
CFM_EVAL_STEPS = 20

CKPT = Path(REPO_ROOT) / 'checkpoints'

MODEL_CLASSES = {
    'redcnn':       REDCNN,
    'dncnn':        DnCNN,
    'unet':         UNet,
    'ctformer':     CTformer,
    'flowmatching': FlowMatching,
}

# Build validation loader (patient-level split, same seed as training)
_, val_p = HDF5CTDataset.split_patients(LOCAL_CACHE, val_fraction=0.2, seed=0)
val_ds   = HDF5CTDataset(LOCAL_CACHE, val_p, patch_size=PATCH, train=False)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

@torch.no_grad()
def eval_model(model, name):
    model.eval()
    # Temporarily reduce ODE steps for CFM to avoid hours-long eval
    _orig = getattr(model, 'num_steps', None)
    if _orig is not None:
        model.num_steps = CFM_EVAL_STEPS

    n, p, s, r, g, nps = 0, 0., 0., 0., 0., 0.
    for i, (low, full) in enumerate(val_loader):
        low, full = low.to(DEVICE), full.to(DEVICE)
        pred = overlapped_inference(model, low, patch_size=PATCH,
                                    margin=PATCH // 4).clamp(0, 1)
        bs = low.size(0)
        p   += psnr(pred, full)      * bs
        s   += ssim(pred, full)      * bs
        r   += rmse(pred, full)      * bs
        g   += gmsd(pred, full)      * bs
        nps += nps_ratio(pred, full) * bs
        n   += bs
        if (i + 1) % 20 == 0:
            print(f'  {name}: {i+1} slices done...')

    if _orig is not None:
        model.num_steps = _orig
    return dict(psnr=p/n, ssim=s/n, rmse=r/n, gmsd=g/n, nps_ratio=nps/n)

rows = []
for name, cls in MODEL_CLASSES.items():
    ckpt_path = CKPT / f'{name}.pt'
    if not ckpt_path.exists():
        print(f'  {name}: checkpoint not found, skipping')
        continue
    print(f'Evaluating {name}...')
    model = (FlowMatching(num_steps=CFM_EVAL_STEPS) if name == 'flowmatching'
             else cls()).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m = eval_model(model, name)
    rows.append((name, m))
    print(f'  PSNR={m["psnr"]:6.3f}  SSIM={m["ssim"]:.4f}  '
          f'RMSE={m["rmse"]:.5f}  GMSD={m["gmsd"]:.5f}  NPS={m["nps_ratio"]:.5f}')

print('\nDone.')

In [ ]:
# Pretty table with pandas
import pandas as pd

if rows:
    df = pd.DataFrame(
        [{"Model": name, **{k: f'{v:.4f}' for k, v in m.items()}} for name, m in rows]
    ).set_index('Model')
    display(df)
else:
    print('No checkpoints found — run Part 2 first.')

---
## Part 4 — Visualise Denoised Outputs

Picks one validation slice and shows the noisy input, full-dose reference,  
and each model's prediction side-by-side.

In [ ]:
import matplotlib.pyplot as plt
import torch, sys
sys.path.insert(0, REPO_ROOT)

from ctdenoiser.data.dataset import HDF5CTDataset
from ctdenoiser.inference import overlapped_inference
from ctdenoiser.models import CTformer, DnCNN, FlowMatching, REDCNN, UNet
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT   = Path(REPO_ROOT) / 'checkpoints'
PATCH  = 64

MODEL_CLASSES = {
    'redcnn':       REDCNN,
    'dncnn':        DnCNN,
    'unet':         UNet,
    'ctformer':     CTformer,
    'flowmatching': FlowMatching,
}

# Load one validation slice
_, val_p = HDF5CTDataset.split_patients(LOCAL_CACHE, val_fraction=0.2, seed=0)
val_ds   = HDF5CTDataset(LOCAL_CACHE, val_p, patch_size=PATCH, train=False)
low, full = val_ds[0]           # first val slice, shape (1, H, W)
low_b  = low.unsqueeze(0).to(DEVICE)   # (1, 1, H, W)
full_np = full.squeeze().cpu().numpy()
low_np  = low.squeeze().cpu().numpy()

# Collect predictions from available checkpoints
preds = {'Noisy (input)': low_np, 'Full-dose (ref)': full_np}
for name, cls in MODEL_CLASSES.items():
    ckpt_path = CKPT / f'{name}.pt'
    if not ckpt_path.exists():
        continue
    model = (FlowMatching(num_steps=20) if name == 'flowmatching' else cls()).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    with torch.no_grad():
        out = overlapped_inference(model, low_b, patch_size=PATCH, margin=PATCH // 4)
    preds[name] = out.squeeze().cpu().numpy()

# Plot
ncols = len(preds)
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))
for ax, (title, img) in zip(axes, preds.items()):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('CTDenoiser — validation slice comparison', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('/content/denoiser_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/denoiser_comparison.png')

In [ ]:
# Zoomed-in crop to highlight texture differences
# Adjust ROW, COL, SIZE to centre on a region of interest
ROW, COL, SIZE = 100, 150, 128

fig, axes = plt.subplots(1, ncols, figsize=(3 * ncols, 3))
for ax, (title, img) in zip(axes, preds.items()):
    crop = img[ROW:ROW+SIZE, COL:COL+SIZE]
    ax.imshow(crop, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.suptitle(f'Crop [{ROW}:{ROW+SIZE}, {COL}:{COL+SIZE}]', fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('/content/denoiser_crop.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5 — Weights & Biases Hyperparameter Sweep

Bayesian search over learning rate, batch size, patch size, **and model architecture**  
(all five models). Each run is logged to your W&B project.

**Before running:** log in to W&B once with `wandb login` or set `WANDB_API_KEY`.

In [ ]:
import wandb
wandb.login()   # prompts for API key if not already set

In [ ]:
SWEEP_PROJECT  = 'ctdenoiser'
SWEEP_COUNT    = 20          # increase for a thorough search
SWEEP_H5_CACHE = LOCAL_CACHE

sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val/psnr', 'goal': 'maximize'},
    'parameters': {
        'model':      {'values': ['redcnn', 'dncnn', 'unet', 'ctformer', 'flowmatching']},
        'lr':         {'values': [1e-5, 1e-4, 1e-3]},
        'batch_size': {'values': [8, 16, 32]},
        'patch_size': {'values': [64, 96]},
        'epochs':     {'value': 1},
        'flow_steps': {'values': [10, 20, 50]},  # only used when model=flowmatching
    },
}

sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f'Sweep created: {sweep_id}')

In [ ]:
import subprocess, sys, os

def train_run():
    """Single W&B sweep run: delegates to train.py so no training logic lives here."""
    run = wandb.init()
    cfg = run.config

    cmd = [
        sys.executable, '-m', 'ctdenoiser.train',
        '--model',           cfg.model,
        '--lr',              str(cfg.lr),
        '--batch-size',      str(cfg.batch_size),
        '--patch-size',      str(cfg.patch_size),
        '--epochs',          str(cfg.epochs),
        '--wandb-project',   SWEEP_PROJECT,
        '--flow-steps-eval', '5',   # always use fast eval in sweep runs
    ]
    if cfg.model == 'flowmatching':
        cmd += ['--flow-steps', str(cfg.flow_steps)]
    if SWEEP_H5_CACHE and os.path.exists(SWEEP_H5_CACHE):
        cmd += ['--h5-cache', SWEEP_H5_CACHE]

    env = {**os.environ, 'WANDB_RUN_ID': run.id, 'WANDB_RESUME': 'must'}
    subprocess.run(cmd, env=env, check=True, cwd=REPO_ROOT)

wandb.agent(sweep_id, function=train_run, count=SWEEP_COUNT)

### Viewing Sweep Results

Open [wandb.ai](https://wandb.ai) → your project `ctdenoiser` → **Sweeps** tab.

Useful views:
- **Parallel coordinates** — see which HP combos dominate on `val/psnr`
- **Scatter plot** `model` vs `val/psnr` — direct architecture comparison
- **Scatter plot** `model` vs `val/gmsd` — perceptual quality comparison  
- **Importance** panel — which hyperparameter matters most per architecture

**Expected pattern:** CNN regressors (RED-CNN, DnCNN) lead on PSNR/SSIM;  
CFM leads on GMSD and NPS-ratio once trained for ≥ 50 epochs.